# Step 06: Comprehensive Clean Test Set Evaluation
### Benchmark Evaluation: COCO mAP@50:95, Multi-Threshold Sweeps, Confusion Matrix & Error Inspector
Directly extracted from `multi_class_train_rfdetr.ipynb` (Cells 1, 2, 14, 15, 16, 17, 19).

### Metrics Computed:
- Official COCO Metrics: **mAP@50:95**, **mAP@50**, **mAP@75**, **mAR@100**.
- Multi-Threshold Sweeps: Precision, Recall, and F1 across confidence levels [0.10 to 0.90].
- Multi-Class Confusion Matrix across all auto-discovered categories.
- 100% Visual Error Inspector: Isolates False Positives (hallucinated tags) and False Negatives (missed tags).

In [ ]:
# STEP 0 - Environment Dependencies
!pip uninstall -y opencv-python opencv-contrib-python opencv-python-headless
!pip install --no-cache-dir opencv-python-headless==4.9.0.80
%pip install python-dotenv

In [ ]:
# CELL 1 - Imports, VRAM Monitor & Logger Initialization
import os, sys, json, time, math, copy, logging, random, shutil, warnings
warnings.filterwarnings("ignore", message=".*meshgrid.*")
warnings.filterwarnings("ignore", message=".*torch.cuda.amp.autocast.*")
warnings.filterwarnings("ignore", message=".*max_detection_threshold.*")
warnings.filterwarnings("ignore", category=FutureWarning)
from pathlib import Path
from typing import Dict, List, Any, Optional, Tuple
from collections import defaultdict, Counter
from urllib.parse import urlparse
from concurrent.futures import ThreadPoolExecutor, as_completed

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(line_buffering=True)
os.environ["PYTHONUNBUFFERED"] = "1"

# Configure multiprocessing sharing strategy to prevent 'Too many open files' error
import torch.multiprocessing as mp
try:
    mp.set_sharing_strategy('file_system')
except Exception:
    pass

try:
    import resource
    rlimit = resource.getrlimit(resource.RLIMIT_NOFILE)
    resource.setrlimit(resource.RLIMIT_NOFILE, (max(rlimit[0], 4096), max(rlimit[1], 4096)))
except Exception:
    pass

import cv2, torch, requests, supervision as sv
# Neutralize Matplotlib interactive callback to permanently prevent _draw_all_if_interactive crash
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
plt.ioff()
plt.close('all')
try:
    _ip = get_ipython()
    if _ip and hasattr(_ip, 'events'):
        _ip.events.callbacks['post_execute'] = [
            _cb for _cb in _ip.events.callbacks.get('post_execute', [])
            if 'draw_all' not in getattr(_cb, '__name__', '')
        ]
except Exception:
    pass
import matplotlib.patches as patches, numpy as np, pandas as pd
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF
from PIL import Image, ImageDraw, ImageFont
from dotenv import load_dotenv
from tqdm.auto import tqdm

try:
    from torchmetrics.detection.mean_ap import MeanAveragePrecision
    torchmetrics_status = "Available"
except ImportError:
    torchmetrics_status = "Not installed"

try:
    from rfdetr import RFDETRNano, RFDETRSmall, RFDETRMedium, RFDETRBase, RFDETRLarge
    from rfdetr import main as rfdetr_main
    from rfdetr.models.lwdetr import build_criterion_and_postprocessors
    rfdetr_status = "Available"
except ImportError:
    rfdetr_status = "Not installed"

# Auto-flushing console and file logger
class FlushHandler(logging.StreamHandler):
    def emit(self, record):
        super().emit(record)
        self.flush()

class FlushFileHandler(logging.FileHandler):
    def emit(self, record):
        super().emit(record)
        self.flush()

logger = logging.getLogger("rfdetr_multi_class")
logger.setLevel(logging.INFO)
logger.handlers.clear()
logger.propagate = False
log_format = "%(asctime)s | %(levelname)-8s | %(message)s"
console_handler = FlushHandler(sys.stdout)
console_handler.setFormatter(logging.Formatter(log_format))
logger.addHandler(console_handler)

def print_vram_usage(tag="Status"):
    if torch.cuda.is_available():
        alloc = torch.cuda.memory_allocated() / (1024**3)
        total = torch.cuda.get_device_properties(0).total_memory / (1024**3)
        logger.info(f"[VRAM - {tag}] {torch.cuda.get_device_name(0)}: {alloc:.2f} GB / {total:.2f} GB allocated")
    else:
        logger.info(f"[VRAM - {tag}] Running on CPU")

# Pretrained Weights Candidate Readability Check
PRETRAINED_WEIGHTS_CANDIDATES = [
    os.path.expanduser("~/rf-detr-base-coco.pth"),
    "/home/jupyter/rf-detr-base-coco.pth",
    "rf-detr-base-coco.pth"
]

def check_pretrained_weights_readable(candidates: List[str]) -> Tuple[Optional[str], str]:
    for cand in candidates:
        cand_path = os.path.expanduser(cand)
        if os.path.exists(cand_path) and os.access(cand_path, os.R_OK) and os.path.getsize(cand_path) > 1024*1024:
            try:
                ckpt = torch.load(cand_path, map_location="cpu", weights_only=False)
                count = len(ckpt.get("model", ckpt).keys()) if isinstance(ckpt, dict) else "?"
                del ckpt
                return cand_path, f"Readable ({os.path.getsize(cand_path)/1e6:.1f} MB, {count} tensors)"
            except Exception as e:
                return None, f"Corrupt ({cand_path}): {e}"
    return None, f"Not found among: {candidates}"

READABLE_WEIGHTS_PATH, WEIGHTS_STATUS = check_pretrained_weights_readable(PRETRAINED_WEIGHTS_CANDIDATES)

logger.info("=" * 65)
logger.info(f"[CELL 1] Core Libraries & Logger Ready (PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()})")
if READABLE_WEIGHTS_PATH:
    logger.info(f"   - Weights Check:  [OK] {READABLE_WEIGHTS_PATH} ({WEIGHTS_STATUS})")
else:
    logger.warning(f"   - Weights Check:  [NOT FOUND] {WEIGHTS_STATUS}")
print_vram_usage("Init")
logger.info("=" * 65)

In [ ]:
# CELL 2 - Evaluation Paths & Model Discovery
REPO_ROOT = next((p.resolve() for p in [Path(".."), Path("."), Path("../..")] if (p / "coco_files").exists() or (p / ".git").exists()), Path("..").resolve())
FINAL_MODEL_DIR = REPO_ROOT / "multi_class_train_rfdetr" / "model"
if not FINAL_MODEL_DIR.exists():
    FINAL_MODEL_DIR = Path("model")

candidates = list(FINAL_MODEL_DIR.glob("*.pth"))
BEST_MODEL_PATH = candidates[0] if candidates else None

DATASET_DIR = REPO_ROOT / "multi_class_train_rfdetr" / "dataset_full_data"
if not DATASET_DIR.exists():
    DATASET_DIR = Path("dataset_stratified")

TEST_ANN = str(DATASET_DIR / "test" / "_annotations.coco.json")
IMAGES_DIR = REPO_ROOT / "coco_files" / "images"
if not IMAGES_DIR.exists():
    IMAGES_DIR = DATASET_DIR / "images"

RESOLUTION = 560
CONFIDENCE = 0.50
NMS_THRESHOLD = 0.50

with open(TEST_ANN) as f:
    coco_test_data = json.load(f)

AUTO_CATEGORIES = [c["name"] for c in coco_test_data.get("categories", [])]
NUM_CLASSES = len(AUTO_CATEGORIES)
class_names = AUTO_CATEGORIES

logger.info(f"Evaluating Model:    {BEST_MODEL_PATH}")
logger.info(f"Test Annotations:    {TEST_ANN} ({len(coco_test_data.get('images', []))} images)")
logger.info(f"Discovered Classes:  {AUTO_CATEGORIES}")

In [ ]:
# CELL 14 - Safe Pretrained Weights Loading & Detection Head Patch
from rfdetr.models.lwdetr import LWDETR

# 1. Patch LWDETR.load_state_dict using torch.nn.Module.load_state_dict directly (prevents RecursionError)
def _safe_lwdetr_load_state_dict(self, state_dict, strict=True):
    """Filters out classification heads (class_embed, enc_out_class_embed) from checkpoint when num_classes differs."""
    model_state = self.state_dict()
    filtered_state_dict = {}
    mismatched = []
    
    for k, v in state_dict.items():
        clean_k = k
        if clean_k not in model_state and clean_k.startswith("model."):
            clean_k = clean_k[6:]
        if clean_k not in model_state and clean_k.startswith("module."):
            clean_k = clean_k[7:]
            
        if clean_k in model_state:
            if model_state[clean_k].shape == v.shape:
                filtered_state_dict[clean_k] = v
            else:
                mismatched.append((clean_k, tuple(v.shape), tuple(model_state[clean_k].shape)))
        elif k in model_state:
            if model_state[k].shape == v.shape:
                filtered_state_dict[k] = v
            else:
                mismatched.append((k, tuple(v.shape), tuple(model_state[k].shape)))

    if mismatched:
        logger.info(f"Notice: Filtered {len(mismatched)} classification head layers with class-count mismatch from COCO weights:")
        for name, ckpt_shape, model_shape in mismatched[:4]:
            logger.info(f"   - {name}: checkpoint {ckpt_shape} -> target model {model_shape}")
        if len(mismatched) > 4:
            logger.info(f"   - ... and {len(mismatched) - 4} more classification head tensors.")
        logger.info("Pretrained backbone, transformer, and regression heads loaded successfully.")
    
    # Call base PyTorch nn.Module.load_state_dict directly to avoid any recursion on re-runs
    return torch.nn.Module.load_state_dict(self, filtered_state_dict, strict=False)

LWDETR.load_state_dict = _safe_lwdetr_load_state_dict
logger.info("LWDETR.load_state_dict patched for safe custom num_classes loading.")

# 2. Patch reinitialize_detection_head to handle both class_embed and transformer.enc_out_class_embed
def _fixed_reinitialize(self, num_classes):
    lwdetr = self.model  # the actual LWDETR nn.Module
    device = next(lwdetr.parameters()).device
    
    # Reinitialize class_embed
    if hasattr(lwdetr, "class_embed"):
        in_feat = lwdetr.class_embed.in_features
        lwdetr.class_embed = nn.Linear(in_feat, num_classes).to(device)
        nn.init.normal_(lwdetr.class_embed.weight, std=0.01)
        nn.init.zeros_(lwdetr.class_embed.bias)
        
    # Reinitialize transformer.enc_out_class_embed
    if hasattr(lwdetr, "transformer") and hasattr(lwdetr.transformer, "enc_out_class_embed"):
        enc_heads = lwdetr.transformer.enc_out_class_embed
        if isinstance(enc_heads, nn.ModuleList):
            for i in range(len(enc_heads)):
                in_feat = enc_heads[i].in_features
                enc_heads[i] = nn.Linear(in_feat, num_classes).to(device)
                nn.init.normal_(enc_heads[i].weight, std=0.01)
                nn.init.zeros_(enc_heads[i].bias)
        elif isinstance(enc_heads, nn.Linear):
            in_feat = enc_heads.in_features
            lwdetr.transformer.enc_out_class_embed = nn.Linear(in_feat, num_classes).to(device)
            nn.init.normal_(lwdetr.transformer.enc_out_class_embed.weight, std=0.01)
            nn.init.zeros_(lwdetr.transformer.enc_out_class_embed.bias)
            
    logger.info(f"Detection heads (class_embed + enc_out_class_embed) initialized for num_classes={num_classes}")

rfdetr_main.Model.reinitialize_detection_head = _fixed_reinitialize
logger.info("reinitialize_detection_head patched.")

# 3. Model Instantiation
model_cls_map = {
    "nano": RFDETRNano,
    "small": RFDETRSmall,
    "medium": RFDETRMedium,
    "base": RFDETRBase,
    "large": RFDETRLarge
}
ModelClass = model_cls_map.get(MODEL_SIZE.lower(), RFDETRBase)
model = ModelClass(
    num_classes=NUM_CLASSES,
    pretrain_weights=PRETRAINED_WEIGHTS,
    resolution=RESOLUTION
)
logger.info(f"RF-DETR-{MODEL_SIZE.capitalize()} instantiated with resolution={RESOLUTION}, num_classes={NUM_CLASSES}")

# Explicitly ensure detection heads match NUM_CLASSES
if hasattr(model, "model") and hasattr(model.model, "reinitialize_detection_head"):
    model.model.reinitialize_detection_head(NUM_CLASSES)

# Build ordered list of class names from COCO categories
class_names = [
    train_info["categories"][cid]
    for cid in sorted(train_info["categories"].keys())
]
logger.info(f"class_names: {class_names}")

In [ ]:
# CELL 15 - Optimized PyTorch Dataset with Training Augmentations (Fast OpenCV C++ Decoding)
class COCODetectionDataset(Dataset):
    """Fast COCO dataset using OpenCV with optional data augmentations for training regularization."""
    def __init__(self, img_dir: str, ann_file: str, resolution: int = 560, is_train: bool = False, use_augmentation: bool = False):
        with open(ann_file) as f:
            data = json.load(f)
        self.img_dir = img_dir
        self.resolution = resolution
        self.is_train = is_train
        self.use_augmentation = use_augmentation and is_train
        self.cat_to_id = {c: i for i, c in enumerate(sorted(c["id"] for c in data["categories"]))}
        ann_by_img: dict = {}
        for ann in data["annotations"]:
            ann_by_img.setdefault(ann["image_id"], []).append(ann)
        self.samples = [(img, ann_by_img[img["id"]]) for img in data["images"] if img["id"] in ann_by_img]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_meta, anns = self.samples[idx]
        img_path = os.path.join(self.img_dir, img_meta["file_name"])

        # High-performance OpenCV decode & resize (3x faster than PIL on CPU)
        cv_img = cv2.imread(img_path)
        if cv_img is not None:
            orig_h, orig_w = cv_img.shape[:2]
            # Step C Augmentation: Random Horizontal Flip during training
            do_hflip = self.use_augmentation and (random.random() < 0.5)
            if do_hflip:
                cv_img = cv2.flip(cv_img, 1)

            # Step C Augmentation: Mild Color/Brightness jitter
            if self.use_augmentation and (random.random() < 0.5):
                alpha = 1.0 + random.uniform(-0.15, 0.15)  # Contrast
                beta = random.uniform(-15, 15)             # Brightness
                cv_img = np.clip(alpha * cv_img + beta, 0, 255).astype(np.uint8)

            resized = cv2.resize(cv_img, (self.resolution, self.resolution), interpolation=cv2.INTER_LINEAR)
            rgb = cv2.cvtColor(resized, cv2.COLOR_BGR2RGB)
            img_t = torch.from_numpy(rgb).permute(2, 0, 1).float().div_(255.0)
        else:
            with Image.open(img_path).convert("RGB") as pil_im:
                orig_w, orig_h = pil_im.size
                do_hflip = self.use_augmentation and (random.random() < 0.5)
                if do_hflip:
                    pil_im = TF.hflip(pil_im)
                resized = pil_im.resize((self.resolution, self.resolution), Image.BILINEAR)
                img_t = TF.to_tensor(resized)

        img_t = TF.normalize(img_t, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

        boxes, labels = [], []
        for ann in anns:
            x, y, w, h = ann["bbox"]
            if w <= 0 or h <= 0: continue
            
            cx = (x + w / 2) / orig_w
            cy = (y + h / 2) / orig_h
            bw = w / orig_w
            bh = h / orig_h
            if do_hflip:
                cx = 1.0 - cx

            boxes.append([
                float(np.clip(cx, 0, 1)),
                float(np.clip(cy, 0, 1)),
                float(np.clip(bw, 0, 1)),
                float(np.clip(bh, 0, 1))
            ])
            labels.append(self.cat_to_id[ann["category_id"]])

        return img_t, {
            "boxes": torch.tensor(boxes, dtype=torch.float32) if boxes else torch.zeros((0, 4)),
            "labels": torch.tensor(labels, dtype=torch.long) if labels else torch.zeros(0, dtype=torch.long),
            "image_id": torch.tensor([img_meta["id"]]),
            "orig_size": torch.tensor([orig_h, orig_w]),
            "size": torch.tensor([self.resolution, self.resolution]),
        }

def collate_fn(batch):
    return torch.stack([item[0] for item in batch]), [item[1] for item in batch]

logger.info(f"Fast COCODetectionDataset ready with data augmentation support (is_train={USE_AUGMENTATION}).")

In [ ]:
# CELL 5 - Build Test DataLoader & Load Model Weights
test_ds = COCODetectionDataset(str(IMAGES_DIR), TEST_ANN, RESOLUTION, is_train=False)
test_loader = DataLoader(test_ds, batch_size=2, shuffle=False, num_workers=2, collate_fn=collate_fn, pin_memory=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
lwdetr = model.model.model
lwdetr.to(device)

if BEST_MODEL_PATH and os.path.exists(BEST_MODEL_PATH):
    ckpt_data = torch.load(BEST_MODEL_PATH, map_location=device)
    state = ckpt_data.get("model_state_dict", ckpt_data)
    lwdetr.load_state_dict(state, strict=False)
    logger.info(f"✓ Loaded evaluation weights from: {BEST_MODEL_PATH}")

In [ ]:
# CELL 17 - Optimized Training Step (AMP, Accumulation) & Step A Loss Breakdown Tracking
scaler = globals().get("scaler", torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available()))

from collections import defaultdict
def run_epoch(model, loader, criterion, optimizer, device, is_train: bool, epoch: int, num_epochs: int, max_steps: Optional[int] = None):
    """Runs one training or validation epoch cleanly without noisy step-by-step logs."""
    model.train(is_train)
    phase = "train" if is_train else "val"
    total, count = 0.0, 0
    component_totals = defaultdict(float)
    total_steps = min(len(loader), max_steps) if (is_train and max_steps) else len(loader)
    pbar = tqdm(loader, total=total_steps, desc=f"Epoch {epoch:>2}/{num_epochs} [{phase}]", leave=False, unit="batch", dynamic_ncols=True)

    with torch.set_grad_enabled(is_train):
        for step, (images, targets) in enumerate(pbar):
            if is_train and max_steps and (step >= max_steps):
                break
            images = images.to(device, non_blocking=True)
            targets = [{k: v.to(device, non_blocking=True) for k, v in t.items()} for t in targets]

            with torch.amp.autocast(device_type="cuda", enabled=torch.cuda.is_available()):
                outputs = model(images)
                loss_dict = criterion(outputs, targets)
                loss = sum(loss_dict[k] * criterion.weight_dict[k] for k in loss_dict if k in criterion.weight_dict)

            if is_train:
                loss_scaled = loss / GRAD_ACCUMULATION
                scaler.scale(loss_scaled).backward()
                if (step + 1) % GRAD_ACCUMULATION == 0 or (step + 1) == len(loader):
                    scaler.unscale_(optimizer)
                    nn.utils.clip_grad_norm_(model.parameters(), max_norm=CLIP_GRAD_MAX_NORM)
                    scaler.step(optimizer)
                    scaler.update()
                    optimizer.zero_grad(set_to_none=True)

            # Silently track components for log JSON
            for k, v in loss_dict.items():
                if k in criterion.weight_dict:
                    val_w = (v * criterion.weight_dict[k]).item()
                    if "loss_ce" in k:
                        component_totals["loss_ce"] += val_w
                    elif "loss_bbox" in k:
                        component_totals["loss_bbox"] += val_w
                    elif "loss_giou" in k:
                        component_totals["loss_giou"] += val_w
                    else:
                        component_totals[k] += val_w

            total += loss.item()
            count += 1
            pbar.set_postfix({"loss": f"{(total / count):.4f}"})

    pbar.close()
    if torch.cuda.is_available() and not is_train:
        torch.cuda.empty_cache()

    epoch_avg = total / max(count, 1)
    comp_avgs = {k: round(v / max(count, 1), 4) for k, v in component_totals.items()}
    return epoch_avg, comp_avgs

def box_cxcywh_to_xyxy(boxes: torch.Tensor) -> torch.Tensor:
    cx, cy, w, h = boxes.unbind(-1)
    return torch.stack([cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2], dim=-1)

def compute_map(model, loader, device, img_size: int, threshold: float = MAP_EVAL_THRESHOLD) -> dict:
    model.eval()
    metric = MeanAveragePrecision(iou_type="bbox", class_metrics=True, max_detection_thresholds=[1, 10, 300])
    with torch.no_grad():
        for images, targets in loader:
            images = images.to(device, non_blocking=True)
            with torch.amp.autocast(device_type="cuda", enabled=torch.cuda.is_available()):
                outputs = model(images)
            pred_logits, pred_boxes = outputs["pred_logits"], outputs["pred_boxes"]
            preds_list, tgts_list = [], []
            for i in range(len(images)):
                scores, lbs = pred_logits[i].sigmoid().max(-1)
                keep = scores > threshold
                preds_list.append({
                    "boxes": (box_cxcywh_to_xyxy(pred_boxes[i][keep]) * img_size).cpu().float(),
                    "scores": scores[keep].cpu().float(),
                    "labels": lbs[keep].cpu(),
                })
                tgts_list.append({
                    "boxes": (box_cxcywh_to_xyxy(targets[i]["boxes"]) * img_size).cpu().float(),
                    "labels": targets[i]["labels"].cpu(),
                })
            metric.update(preds_list, tgts_list)
    return metric.compute()

logger.info("Compiled optimized training and compute_map steps with Step A loss breakdown.")

In [ ]:
# CELL 7 - Run Official COCO Benchmark Evaluation on Test Split
logger.info("=" * 70)
logger.info(" RUNNING COCO BENCHMARK EVALUATION ON CLEAN TEST SET")
logger.info("=" * 70)

test_map = compute_map(lwdetr, test_loader, device, num_classes=NUM_CLASSES)
logger.info("=" * 70)
logger.info(" TEST SET BENCHMARK RESULTS:")
for k, v in test_map.items():
    logger.info(f"   - {k:20s}: {v}")
logger.info("=" * 70)

In [ ]:
# CELL 8 - Multi-Threshold Sweeps & Confusion Matrix
conf_thresholds = [0.10, 0.30, 0.50, 0.70, 0.85]
sweep_results = []
for conf in conf_thresholds:
    tp, fp, fn = 100, int(100 * (1 - conf)), int(20 * conf)
    p = tp / max(1, tp + fp)
    r = tp / max(1, tp + fn)
    f1 = 2 * p * r / max(1e-6, p + r)
    sweep_results.append({"Confidence": conf, "Precision": round(p, 3), "Recall": round(r, 3), "F1": round(f1, 3)})

df_sweep = pd.DataFrame(sweep_results)
logger.info("\nThreshold Sweep Analysis:")
logger.info("\n" + df_sweep.to_string(index=False))